**Stage 1: multi-person pose estimation + tracking.** YOLOv8x-Pose (17 COCO joints) with a centroid tracker (new ID if distance >= 200 px). Writes `<vid>-pose_estimations.json` consumed by `Multi_Pose_Overlay_Gender.ipynb`.

In [ ]:
"""
Pose Estimator
- Runs YOLOv8x-pose on input video
- Outputs: JSON with per-frame, per-person 17-joint keypoints
           Annotated video with skeleton overlay
- JSON format: List[frames] → List[persons] → {person_id, keypoints[{x,y,score}]}
"""

!pip install ultralytics

import cv2
import numpy as np
import json
from ultralytics import YOLO

# ── Paths ─────────────────────────────────────────────────────────────────────
video_path            = '/content/vid9.avi'       # ← change per video
output_json_path      = '/content/vid9-pose_estimations.json'
output_video_path     = '/content/vid9-pose_annotated.mp4'
# ─────────────────────────────────────────────────────────────────────────────

# 17 COCO keypoint names (YOLOv8 order)
KEYPOINT_NAMES = [
    'nose',           # 0
    'left_eye',       # 1
    'right_eye',      # 2
    'left_ear',       # 3
    'right_ear',      # 4
    'left_shoulder',  # 5
    'right_shoulder', # 6
    'left_elbow',     # 7
    'right_elbow',    # 8
    'left_wrist',     # 9
    'right_wrist',    # 10
    'left_hip',       # 11
    'right_hip',      # 12
    'left_knee',      # 13
    'right_knee',     # 14
    'left_ankle',     # 15
    'right_ankle',    # 16
]

# 17-joint COCO skeleton connections
SKELETON_17 = [
    (0, 1), (0, 2),             # Nose → eyes
    (1, 3), (2, 4),             # Eyes → ears
    (5, 6),                     # Shoulders
    (5, 7), (7, 9),             # Left arm
    (6, 8), (8, 10),            # Right arm
    (5, 11), (6, 12),           # Shoulders → hips
    (11, 12),                   # Hips
    (11, 13), (13, 15),         # Left leg
    (12, 14), (14, 16),         # Right leg
]

KEYPOINT_COLORS = [
    (0, 255, 255),  # nose
    (0, 255, 0),    # left_eye
    (0, 255, 0),    # right_eye
    (255, 255, 0),  # left_ear
    (255, 255, 0),  # right_ear
    (255, 0, 0),    # left_shoulder
    (0, 0, 255),    # right_shoulder
    (255, 0, 0),    # left_elbow
    (0, 0, 255),    # right_elbow
    (255, 0, 0),    # left_wrist
    (0, 0, 255),    # right_wrist
    (255, 0, 128),  # left_hip
    (0, 128, 255),  # right_hip
    (255, 0, 128),  # left_knee
    (0, 128, 255),  # right_knee
    (255, 0, 128),  # left_ankle
    (0, 128, 255),  # right_ankle
]

CONF_THRESHOLD = 0.5


def assign_person_ids(frame_persons, prev_persons, next_id):
    if not prev_persons:
        for i, p in enumerate(frame_persons):
            p['person_id'] = next_id + i
        return frame_persons, next_id + len(frame_persons)

    def centroid(kps):
        valid = [kp for kp in kps if kp['score'] > 0.3]
        if not valid:
            return None
        return (np.mean([k['x'] for k in valid]),
                np.mean([k['y'] for k in valid]))

    used_prev = set()
    for p in frame_persons:
        c = centroid(p['keypoints'])
        best_dist, best_id = float('inf'), None
        for prev in prev_persons:
            if prev['person_id'] in used_prev:
                continue
            pc = centroid(prev['keypoints'])
            if c and pc:
                d = np.hypot(c[0] - pc[0], c[1] - pc[1])
                if d < best_dist and d < 200:   # increased from 120 → 200
                    best_dist, best_id = d, prev['person_id']

        if best_id is not None:
            p['person_id'] = best_id
            used_prev.add(best_id)
        else:
            p['person_id'] = next_id
            next_id += 1

    return frame_persons, next_id


# ── Load model and video ──────────────────────────────────────────────────────
print("Loading YOLOv8x-pose model...")
model = YOLO('yolov8x-pose.pt')

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {video_path}")

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {frame_width}x{frame_height}, FPS={fps}, Frames={total_frames}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

all_frames_data = []
prev_persons    = []
next_id         = 0
frame_idx       = 0

print("Processing frames...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    frame_persons = []

    for r in results:
        if r.keypoints is None or r.keypoints.data.numel() == 0:
            continue
        for kpt_data in r.keypoints.data.cpu().numpy():
            keypoints = []
            for x, y, score in kpt_data:
                keypoints.append({'x': float(x), 'y': float(y), 'score': float(score)})
            frame_persons.append({'person_id': -1, 'keypoints': keypoints})

    # Assign consistent IDs across frames
    frame_persons, next_id = assign_person_ids(frame_persons, prev_persons, next_id)
    prev_persons = frame_persons

    # Draw skeleton on frame (neutral color — gender overlay is separate script)
    for person in frame_persons:
        kps = person['keypoints']
        pid = person['person_id']

        # Draw bones
        for p1, p2 in SKELETON_17:
            if p1 >= len(kps) or p2 >= len(kps):
                continue
            k1, k2 = kps[p1], kps[p2]
            if k1['score'] > CONF_THRESHOLD and k2['score'] > CONF_THRESHOLD:
                pt1 = (int(k1['x']), int(k1['y']))
                pt2 = (int(k2['x']), int(k2['y']))
                cv2.line(frame, pt1, pt2, (255, 255, 255), 4, cv2.LINE_AA)
                cv2.line(frame, pt1, pt2, (0, 200, 200), 2, cv2.LINE_AA)

        # Draw joints
        for i, kp in enumerate(kps):
            if kp['score'] > CONF_THRESHOLD:
                center = (int(kp['x']), int(kp['y']))
                color  = KEYPOINT_COLORS[i % len(KEYPOINT_COLORS)]
                cv2.circle(frame, center, 7,  (255, 255, 255), -1)
                cv2.circle(frame, center, 5,  color,           -1)

        # Draw person ID label
        valid_kps = [k for k in kps if k['score'] > CONF_THRESHOLD]
        if valid_kps:
            top_y = int(min(k['y'] for k in valid_kps)) - 10
            top_x = int(np.mean([k['x'] for k in valid_kps]))
            cv2.putText(frame, f"ID:{pid}", (top_x - 15, max(top_y, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    writer.write(frame)
    all_frames_data.append(frame_persons)
    frame_idx += 1

    if frame_idx % 25 == 0:
        print(f"  {frame_idx}/{total_frames} frames | {len(frame_persons)} person(s) detected")

cap.release()
writer.release()

# Save JSON
def make_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)): return float(obj)
    if isinstance(obj, (np.int32,  np.int64)):    return int(obj)
    raise TypeError(f"Not serializable: {type(obj)}")

with open(output_json_path, 'w') as f:
    json.dump(all_frames_data, f, default=make_serializable, indent=2)

print(f"\n✓ JSON saved  → {output_json_path}")
print(f"✓ Video saved → {output_video_path}")
print(f"  Total frames processed: {frame_idx}")
print(f"  Total unique person IDs assigned: {next_id}")
print(f"\nPerson IDs seen: 0 to {next_id - 1}")
print("Use these IDs in the gender assignment dict in pose_overlay_gender.py")

from IPython.display import Video
Video(output_video_path, embed=True, width=720)